In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import chi2


In [ ]:
#perform linear fit with offset and with constant function at start before the linear rise 
def linear_fit(x, a, b):
    return a * x + b 
# power is 0.0 microW with shutter on for the laser

Laser_current = np.array([10, 20, 30, 40,50,55, 60,65, 70,75, 80, 90,100, 110, 120, 130, 140, 150, 160, 180, 200,220, 240,250, 260,270, 280]) #mA
Laser_power = np.array([2.3, 6, 10.5, 16.7, 25.6,  31.7, 2815,4993, 7220, 10020, 13820,18600,25030, 30590, 35430, 42280, 46920, 53400, 60100, 71100, 82700,  95400, 104800, 109300, 114700, 122700, 125900   ])  # data in microW
popt, pcov = curve_fit(linear_fit, Laser_current, Laser_power)
a_fit, b_fit = popt
a_err, b_err = np.sqrt(np.diag(pcov))
print(f"Fitted parameters: a = {a_fit:.4f} ± {a_err:.4f}, b = {b_fit:.4f} ± {b_err:.4f}")
#reduced chi squared and p-value
y_fit = linear_fit(Laser_current, *popt)
chi2_val = np.sum(((Laser_power - y_fit) / np.sqrt(Laser_power))**2)
ndf = len(Laser_current) - len(popt)
red_chi2 = chi2_val / ndf if ndf > 0 else np.nan
pval = 1.0 - chi2.cdf(chi2_val, ndf) if ndf > 0 else np.nan
print(f"Reduced chi²: {red_chi2:.3f}, p-value: {pval:.3e}")

plt.plot(Laser_current, Laser_power, 'o', label='Data', color = "maroon")
x_smooth = np.linspace(Laser_current.min(), Laser_current.max(), 100)
plt.plot(x_smooth, linear_fit(x_smooth, *popt), label='Linear Fit', color = "green")
plt.xlabel('Laser Current (A)')
plt.ylabel('Laser Power (W)')
plt.title('Laser Power vs. Current with Linear Fit')
plt.legend()
plt.show()

In [ ]:
#we fit a function of the form a* (sin(x)/x)^2 +c where x = b(T-T_0)
def sinc_squared(T, a, b, T0, c):
    x_scaled = b * (T - T0)
    sinc = np.sinc(x_scaled / np.pi)  # np.sinc is sin(pi*x)/(pi*x)
    return a * sinc**2 + c

T = np.array([ 31.5, 32, 32.5, 33, 33.5, 34, 34.5, 35, 35.5, 36, 36.5, 37.5, 38, 38.5, 39, 39.5, 40]) # Temperature in degree Celsius
T_err = np.array([0.1, 0.1, 0.1,0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]) # Error in temperature
power = np.array([0.01, 0.02,    ]) # Power in microWatt
power_err = 0.01*power # 1 percent of power is error
popt, pcov = curve_fit(sinc_squared, T, power, sigma=power_err, absolute_sigma=True,)
a_fit, b_fit, T0_fit, c_fit = popt
a_err, b_err, T0_err, c_err = np.sqrt(np.diag(pcov))
print(f"Fitted parameters:")
print(f"a = {a_fit:.4f} ± {a_err:.4f}")
print(f"b = {b_fit:.4f} ± {b_err:.4f}")
print(f"T0 = {T0_fit:.4f} ± {T0_err:.4f}")
print(f"c = {c_fit:.4f} ± {c_err:.4f}")
#good of fit reduced chi squared and p value
y_fit = sinc_squared(T, *popt)
chi2_val = np.sum(((power - y_fit) / power_err)**2)
ndf = len(T) - len(popt)
red_chi2 = chi2_val / ndf if ndf > 0 else np.nan
pval = 1.0 - chi2.cdf(chi2_val, ndf) if ndf > 0 else np.nan
print(f"Reduced chi²: {red_chi2:.3f}, p-value: {pval:.3e}")
T_smooth = np.linspace(T.min(), T.max(), 200)
plt.errorbar(T, power, yerr=power_err, xerr=T_err, fmt='o', label='Data', color = "maroon")
plt.plot(T_smooth, sinc_squared(T_smooth, *popt), label='Sinc² Fit', color = "green")
plt.xlabel('Temperature (°C)') 
plt.ylabel('Power (microW)')
plt.title('Power vs. Temperature with Sinc² Fit')
plt.legend()
plt.show()

In [ ]:
angles = np.array([295, 290, 285, 280, 275, 270, 265, 260, 255, 250, 245, 240, 235, 230, 225, 220, 215, 210,205, 200, 195, 190, 185, 180, 175, 170, 165, 160, 155,150 , 145, 140, 135, 130, 125, 120, 115, 110, 105, 100, 95, 90]) # angles in degrees
power = np.array([55.4, 54.2, 46.2, 33.6, 21.2, 10.4, 4.2, 1.1, 0.2, 0.1, 0.1,0.8, 3.3, 9.3, 17.3,  30.2, 41.0, 51.4, 54.8, 52.7, 44.3, 32.9, 20.5,  11.2, 4.5, 1.3, 0.2, 0.1,0.2, 0.8 , 3.1, 9.1 , 17.6, 29.5, 41.4, 51.2, 54.8, 52.1,  44.2, 31.6,20.6, 11.1 ]) # power in microwatt
power_err = 0.01*power # 1 percent error in power
#fit cos^4 function of the form a*cos^4(b*(theta-theta0)) + c
def cos4(theta, a, b, theta0, c):
    x_scaled = b * (theta - theta0)
    return a * np.cos(np.radians(x_scaled))**4 + c

#fit also sin^2 * cos^2 function of the form a*sin^2(b*(theta-theta0))*cos^2(b*(theta-theta0)) + c
def sin2_cos2(theta, a, b, theta0, c):
    x_scaled = b * (theta - theta0)
    return a * np.sin(np.radians(x_scaled))**2 * np.cos(np.radians(x_scaled))**2 + c
popt_cos4, pcov_cos4 = curve_fit(cos4, angles, power, sigma=power_err, absolute_sigma=True)
a_cos4, b_cos4, theta0_cos4, c_cos4 = popt_cos4
a_cos4_err, b_cos4_err, theta0_cos4_err, c_cos4_err = np.sqrt(np.diag(pcov_cos4))
print(f"Cos^4 Fit parameters:") 
#reduced chi squared and p value for cos^4 fit
y_fit_cos4 = cos4(angles, *popt_cos4)
chi2_val_cos4 = np.sum(((power - y_fit_cos4) / power_err)**2)
ndf_cos4 = len(angles) - len(popt_cos4)
plt.errorbar(angles, power, yerr=power_err,  fmt='o', label='Data', color = "maroon")
angles_smooth = np.linspace(angles.min(), angles.max(), 50)
plt.plot(angles_smooth, cos4(T_smooth, *popt), label='cos4 Fit', color = "green")

plt.legend()
plt.show()





## Measurement after 18:30 with Polarizing Beam Splitter


In [ ]:
angles = np.array([300,  290,  280,  270,  260,250,  240, 230, 220,  210,200,  190, 180, 170, 160, 150, 140, 130, 120, 110, 100]) # angles in degrees
power = np.array([35.9, 33.7,  19,5.3, 0.4, 0.0, 0.8,  7.5, 23.3, 36.3, 33.9, 18.6, 5, 0.3, 0.0, 0.9, 7.8, 23.1, 36.5, 34.2, 19.3  ]) # power in microwatt
power_err = 0.01*power # 1 percent error in power
plt.plot(angles, power, 'o', label='Data', color='red')

## 12.03.2026

In [ ]:
import numpy as np  
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

angles = np.array([320, 317.5, 315, 312.5, 310, 307.5, 305, 302.5, 300, 297.5, 295, 292.5, 290, 287.5, 285, 282.5, 280, 277.5, 275, 272.5, 270, 267.5, 265, 262.5, 260, 257.5, 255, 252.5, 250, 247.5, 245, 242.5, 240, 237.5, 235, 232.5,  230 , 227.5, 225, 225.5, 220, 217.5, 215, 212.5, 210, 207.5, 205, 202.5, 200, 197.5, 195, 192.5, 190 , 187.5, 185, 182.5 , 180, 177.5, 175 , 172.5, 170, 167.5, 165, 162.5, 160]) # angles in degrees
fundamental_power = np.array([ 58.9,68.1, 79.3, 86.1, 96.0 ,103.3, 110.4,114.8,117.9, 118.6, 117.7, 115, 110.7, 105.1, 97.7, 88.8, 79.1, 69.5, 60.7, 48.8,39.5, 31.1, 21.5, 15, 8.5, 4.5, 1.6, 0.7, 1.9, 3.8, 8.8, 14.3, 21.3 , 30.7 , 38.1, 48.6, 58.9, 68.6, 77.8, 87.4, 97.2, 104.3, 110.2, 114.5, 117.3, 118.1, 117.5, 114.8, 110.3, 103.7, 96.9, 89.7, 79.0, 71.2, 60.7, 49.3, 39.2, 29.1, 21.7, 14.1, 8.2, 3.8, 1.3, 0.3, 1.2 ]) # power in milliwatt
power_err = 0.01*fundamental_power # 1 percent error in power
angles_err = 1 # or 0.5 degrees
plt.errorbar(angles, fundamental_power, xerr=angles_err, yerr=power_err, fmt='o', capsize=5)

In [ ]:
print(len(angles))
len(fundamental_power)

## Temperature Depedendence of the crystal

In [ ]:
Temperature = np.array([ 39.9, 39.5, 39, 38.5, 38, 37.8, 37.5,37.2, 37, 36.8, 36.5,36.2, 36, 35.8 , 35.5, 35.2, 35, 34.5, 34, 33.5, 33, 32.5, 32, 31.5, 31, 30.5, 30, 29.5, 29, 28.5, 28, 27.5, 27 ]) # Temperature in degree Celsius
Power_2HG = np.array([ 0.16, 0.11, 0.98, 0.49, 5, 8.6, 16.9, 29.7, 33.5, 34.3, 28.3, 18, 11.4, 5, 1.43, 0.33, 0.25, 0.21, 0.25, 0.1, 0.04, 0.1, 0.05, 0.03,  0.06, 0.02, 0.02, 0.01, 0.03, 0.02, 0.02, 0.02,  0.03 ]) # power in microwatt
Power_err = Power_2HG * 0.01 # 1 percent error in power because large power fluctuations
Temperature_err = 0.1 # degree Celsius
# plt.errorbar(Temperature, Power_2HG, xerr=Temperature_err, yerr=Power_err, fmt='o', capsize=5, color = "green")
#fit sinc function of the form a * sinc(b*(T-T0))^2 + c
def sinc_squared(T, a, b, T0, c):
    x_scaled = b * (T - T0)
    sinc = np.sinc(x_scaled / np.pi)  # np.sinc is sin(pi*x)/(pi*x)
    return a * sinc**2 + c
popt, pcov = curve_fit(sinc_squared, Temperature, Power_2HG, sigma=Power_err, absolute_sigma=True)
a_fit, b_fit, T0_fit, c_fit = popt
a_err, b_err, T0_err, c_err = np.sqrt(np.diag(pcov))
print(f"Fitted parameters:")
print(f"a = {a_fit:.4f} ± {a_err:.4f}")
print(f"b = {b_fit:.4f} ± {b_err:.4f}")
print(f"T0 = {T0_fit:.4f} ± {T0_err:.4f}")
print(f"c = {c_fit:.4f} ± {c_err:.4f}")
#good of fit reduced chi squared and p value
y_fit = sinc_squared(Temperature, *popt)
chi2_val = np.sum(((Power_2HG - y_fit) / Power_err)**2)
ndf = len(Temperature) - len(popt)
chi2 = chi2_val
red_chi2 = chi2_val / ndf if ndf > 0 else np.nan
# pval = 1.0 - chi2.cdf(chi2_val, ndf) if ndf > 0 else np.nan
print(f"Reduced chi²: {red_chi2:.3f}")
T_smooth = np.linspace(Temperature.min(), Temperature.max(), 200)
plt.errorbar(Temperature, Power_2HG, xerr=Temperature_err, yerr=Power_err, fmt='o', capsize=5, label='Data', color = "green")
plt.plot(T_smooth, sinc_squared(T_smooth, *popt), label='Sinc² Fit', color = "blue")
plt.xlabel('Temperature (°C)')
plt.ylabel('Power (microW)')
plt.title('Power vs. Temperature with Sinc² Fit')
plt.legend()
plt.show()